# Network Activity Monitor Demo

This notebook demonstrates the Algorand Network Activity Monitor for interest rate determination based on network health and ecosystem activity.

## Features Demonstrated
- Network health monitoring (TPS, consensus, block times)
- DApp activity analysis (DeFi, NFT, governance)
- Ecosystem vitality assessment
- Activity-based rate adjustments
- Real-time network monitoring

In [ ]:
# Import required libraries
import sys
import asyncio
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datetime import datetime, timedelta
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Add the parent directory to the path
sys.path.append('../')

# Import the network activity engine
from network_activity_monitor.core.activity_engine import NetworkActivityEngine

# Setup plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("plasma")
%matplotlib inline

## Configuration Setup

Configure the network activity monitoring engine.

In [ ]:
# Configuration for the network activity engine
config = {
    'network_monitoring': {
        'health_weights': {
            'transaction_throughput': 0.30,
            'consensus_health': 0.25,
            'node_participation': 0.20,
            'mempool_status': 0.25
        },
        'dapp_weights': {
            'smart_contract_activity': 0.35,
            'defi_protocol_usage': 0.30,
            'nft_marketplace_activity': 0.15,
            'governance_activity': 0.20
        }
    },
    'network_thresholds': {
        'tps_levels': {
            'excellent': 1000,
            'good': 500,
            'average': 100,
            'poor': 50,
            'critical': 10
        },
        'block_time': {
            'target': 4.5,
            'excellent_variance': 0.5,
            'good_variance': 1.0,
            'poor_variance': 2.0
        },
        'participation_levels': {
            'excellent': 0.95,
            'good': 0.90,
            'average': 0.85,
            'poor': 0.80,
            'critical': 0.75
        }
    },
    'utilization_categories': {
        'very_high': {
            'min_tps': 800,
            'description': 'Network operating at high capacity',
            'rate_impact': -0.1
        },
        'high': {
            'min_tps': 400,
            'description': 'Strong network activity',
            'rate_impact': -0.05
        },
        'moderate': {
            'min_tps': 100,
            'description': 'Normal network activity',
            'rate_impact': 0.0
        },
        'low': {
            'min_tps': 50,
            'description': 'Below average activity',
            'rate_impact': 0.05
        },
        'very_low': {
            'min_tps': 0,
            'description': 'Minimal network activity',
            'rate_impact': 0.1
        }
    },
    'dapp_monitoring': {
        'defi_protocols': {
            'tinyman': {
                'app_ids': [552635992, 624956175],
                'metrics': ['swaps', 'liquidity', 'volume']
            },
            'algofi': {
                'app_ids': [465818260, 465818263],
                'metrics': ['borrows', 'supplies', 'liquidations']
            }
        },
        'nft_platforms': {
            'algogems': {
                'app_ids': [403380490],
                'metrics': ['sales', 'listings', 'volume']
            }
        },
        'activity_levels': {
            'high_activity': 10000,
            'moderate_activity': 1000,
            'low_activity': 100,
            'minimal_activity': 10
        }
    },
    'algorand_config': {
        'node': {'url': 'https://mainnet-api.algonode.cloud'},
        'indexer': {'url': 'https://mainnet-idx.algonode.cloud'}
    }
}

# Initialize the activity engine
activity_engine = NetworkActivityEngine(config_dict=config)
print("✅ Network Activity Engine initialized successfully")

## Network Health Simulation

Let's simulate network health monitoring over time.

In [ ]:
# Generate mock network activity data over time
def generate_network_activity_timeline(hours=24, interval_minutes=30):
    """Generate simulated network activity data over time"""
    import random
    
    timestamps = []
    current_time = datetime.utcnow() - timedelta(hours=hours)
    
    while current_time <= datetime.utcnow():
        timestamps.append(current_time)
        current_time += timedelta(minutes=interval_minutes)
    
    data = []
    
    for i, timestamp in enumerate(timestamps):
        # Simulate daily patterns (higher activity during business hours)
        hour = timestamp.hour
        
        # Base activity with daily patterns
        if 6 <= hour <= 12:  # Morning activity
            base_multiplier = random.uniform(1.2, 1.8)
        elif 13 <= hour <= 20:  # Peak activity
            base_multiplier = random.uniform(1.5, 2.2)
        elif 21 <= hour <= 23:  # Evening activity
            base_multiplier = random.uniform(1.0, 1.5)
        else:  # Low activity
            base_multiplier = random.uniform(0.5, 1.0)
        
        # Add some randomness and trends
        trend_factor = 1 + (i / len(timestamps)) * 0.2  # Slight upward trend
        noise = random.uniform(0.8, 1.2)
        
        # Network metrics
        base_tps = 150
        tps = max(10, base_tps * base_multiplier * trend_factor * noise)
        
        block_time = random.uniform(4.2, 4.8) + random.uniform(-0.5, 0.5)
        participation = random.uniform(0.88, 0.95)
        mempool_util = random.uniform(0.05, 0.4)
        
        # DApp activity
        defi_volume = random.uniform(500000, 5000000) * base_multiplier
        dapp_calls = int(random.uniform(5000, 50000) * base_multiplier)
        nft_sales = int(random.uniform(20, 200) * base_multiplier)
        governance_votes = int(random.uniform(100, 1000))
        
        # Calculate activity scores
        network_score = min(1.0, tps / 1000) * 0.4 + (1 - mempool_util) * 0.3 + participation * 0.3
        dapp_score = min(1.0, defi_volume / 5000000) * 0.4 + min(1.0, dapp_calls / 50000) * 0.6
        overall_score = network_score * 0.6 + dapp_score * 0.4
        
        # Determine utilization category
        if tps >= 800:
            category, rate_impact = "very_high", -0.1
        elif tps >= 400:
            category, rate_impact = "high", -0.05
        elif tps >= 100:
            category, rate_impact = "moderate", 0.0
        elif tps >= 50:
            category, rate_impact = "low", 0.05
        else:
            category, rate_impact = "very_low", 0.1
        
        data.append({
            'timestamp': timestamp,
            'tps': tps,
            'block_time': block_time,
            'participation_rate': participation,
            'mempool_utilization': mempool_util,
            'defi_volume_24h': defi_volume,
            'dapp_calls': dapp_calls,
            'nft_sales': nft_sales,
            'governance_votes': governance_votes,
            'network_score': network_score,
            'dapp_score': dapp_score,
            'overall_activity_score': overall_score,
            'utilization_category': category,
            'rate_adjustment': rate_impact
        })
    
    return pd.DataFrame(data)

# Generate 24 hours of network activity data
activity_data = generate_network_activity_timeline(hours=24, interval_minutes=15)
print(f"📊 Generated {len(activity_data)} network activity data points over 24 hours")
print(f"📈 Average TPS: {activity_data['tps'].mean():.1f}")
print(f"🎯 Average Activity Score: {activity_data['overall_activity_score'].mean():.3f}")
print(f"⚡ Peak TPS: {activity_data['tps'].max():.1f}")
print(f"📉 Minimum TPS: {activity_data['tps'].min():.1f}")

## Network Activity Visualization

In [ ]:
# Create comprehensive network activity dashboard
fig, axes = plt.subplots(3, 2, figsize=(16, 14))
fig.suptitle('Algorand Network Activity Monitoring Dashboard', fontsize=16, fontweight='bold')

# 1. TPS Over Time
axes[0, 0].plot(activity_data['timestamp'], activity_data['tps'], linewidth=2, color='blue')
axes[0, 0].axhline(y=config['network_thresholds']['tps_levels']['excellent'], 
                   color='green', linestyle='--', alpha=0.7, label='Excellent (1000)')
axes[0, 0].axhline(y=config['network_thresholds']['tps_levels']['good'], 
                   color='orange', linestyle='--', alpha=0.7, label='Good (500)')
axes[0, 0].axhline(y=config['network_thresholds']['tps_levels']['average'], 
                   color='red', linestyle='--', alpha=0.7, label='Average (100)')
axes[0, 0].set_title('Transactions Per Second (TPS)')
axes[0, 0].set_ylabel('TPS')
axes[0, 0].legend()
axes[0, 0].tick_params(axis='x', rotation=45)

# 2. Block Time Variance
axes[0, 1].plot(activity_data['timestamp'], activity_data['block_time'], linewidth=2, color='purple')
axes[0, 1].axhline(y=config['network_thresholds']['block_time']['target'], 
                   color='green', linestyle='-', alpha=0.7, label='Target (4.5s)')
axes[0, 1].fill_between(activity_data['timestamp'], 
                        config['network_thresholds']['block_time']['target'] - 0.5,
                        config['network_thresholds']['block_time']['target'] + 0.5,
                        alpha=0.2, color='green', label='Excellent Range')
axes[0, 1].set_title('Block Time Performance')
axes[0, 1].set_ylabel('Block Time (seconds)')
axes[0, 1].legend()
axes[0, 1].tick_params(axis='x', rotation=45)

# 3. Network Health Metrics
axes[1, 0].plot(activity_data['timestamp'], activity_data['participation_rate'], 
                linewidth=2, color='green', label='Participation Rate')
axes[1, 0].plot(activity_data['timestamp'], activity_data['mempool_utilization'], 
                linewidth=2, color='red', label='Mempool Utilization')
axes[1, 0].set_title('Network Health Indicators')
axes[1, 0].set_ylabel('Rate/Utilization')
axes[1, 0].legend()
axes[1, 0].tick_params(axis='x', rotation=45)

# 4. DApp Activity
ax2 = axes[1, 1].twinx()
line1 = axes[1, 1].plot(activity_data['timestamp'], activity_data['defi_volume_24h'] / 1e6, 
                        linewidth=2, color='gold', label='DeFi Volume ($M)')
line2 = ax2.plot(activity_data['timestamp'], activity_data['dapp_calls'], 
                 linewidth=2, color='cyan', label='DApp Calls')
axes[1, 1].set_title('DApp Ecosystem Activity')
axes[1, 1].set_ylabel('DeFi Volume ($M)', color='gold')
ax2.set_ylabel('DApp Calls', color='cyan')
axes[1, 1].tick_params(axis='x', rotation=45)

# Combine legends
lines1, labels1 = axes[1, 1].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[1, 1].legend(lines1 + lines2, labels1 + labels2, loc='upper left')

# 5. Activity Scores
axes[2, 0].plot(activity_data['timestamp'], activity_data['network_score'], 
                linewidth=2, color='blue', label='Network Score')
axes[2, 0].plot(activity_data['timestamp'], activity_data['dapp_score'], 
                linewidth=2, color='orange', label='DApp Score')
axes[2, 0].plot(activity_data['timestamp'], activity_data['overall_activity_score'], 
                linewidth=3, color='red', label='Overall Score')
axes[2, 0].set_title('Activity Scores')
axes[2, 0].set_ylabel('Score (0-1)')
axes[2, 0].legend()
axes[2, 0].tick_params(axis='x', rotation=45)

# 6. Rate Adjustment Impact
rate_colors = ['green' if x < 0 else 'red' if x > 0 else 'gray' for x in activity_data['rate_adjustment']]
axes[2, 1].scatter(activity_data['timestamp'], activity_data['rate_adjustment'], 
                   c=rate_colors, alpha=0.7, s=30)
axes[2, 1].axhline(y=0, color='black', linestyle='-', alpha=0.5)
axes[2, 1].set_title('Interest Rate Adjustments')
axes[2, 1].set_ylabel('Rate Adjustment (%)')
axes[2, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Activity Category Analysis

In [ ]:
# Analyze utilization categories over time
category_analysis = activity_data.groupby('utilization_category').agg({
    'timestamp': 'count',
    'tps': ['mean', 'min', 'max'],
    'overall_activity_score': 'mean',
    'rate_adjustment': 'mean'
}).round(3)

category_analysis.columns = ['Count', 'Avg_TPS', 'Min_TPS', 'Max_TPS', 'Avg_Score', 'Avg_Rate_Adj']

print("📊 Network Utilization Category Analysis")
print("=" * 60)
print(category_analysis)

# Calculate time distribution
category_distribution = activity_data['utilization_category'].value_counts()
total_periods = len(activity_data)

print("\n⏰ Time Distribution by Category:")
for category, count in category_distribution.items():
    percentage = (count / total_periods) * 100
    description = config['utilization_categories'][category]['description']
    rate_impact = config['utilization_categories'][category]['rate_impact']
    print(f"{category.upper():<12}: {count:>3} periods ({percentage:>5.1f}%) - {description} (Rate: {rate_impact:+.2f}%)")

In [ ]:
# Create utilization category visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Network Utilization Analysis', fontsize=16, fontweight='bold')

# 1. Category Distribution Pie Chart
category_counts = activity_data['utilization_category'].value_counts()
colors = ['darkgreen', 'green', 'yellow', 'orange', 'red']
axes[0, 0].pie(category_counts.values, labels=category_counts.index, autopct='%1.1f%%', 
               startangle=90, colors=colors[:len(category_counts)])
axes[0, 0].set_title('Time Distribution by Utilization Category')

# 2. TPS Distribution by Category
category_order = ['very_high', 'high', 'moderate', 'low', 'very_low']
existing_categories = [cat for cat in category_order if cat in activity_data['utilization_category'].unique()]

box_data = [activity_data[activity_data['utilization_category'] == cat]['tps'] for cat in existing_categories]
axes[0, 1].boxplot(box_data, labels=existing_categories)
axes[0, 1].set_title('TPS Distribution by Category')
axes[0, 1].set_ylabel('TPS')
axes[0, 1].tick_params(axis='x', rotation=45)

# 3. Activity Score vs TPS Scatter
scatter = axes[1, 0].scatter(activity_data['tps'], activity_data['overall_activity_score'], 
                           c=activity_data['rate_adjustment'], cmap='RdYlGn_r', 
                           alpha=0.7, s=50)
axes[1, 0].set_xlabel('TPS')
axes[1, 0].set_ylabel('Overall Activity Score')
axes[1, 0].set_title('Activity Score vs TPS (colored by rate adjustment)')
plt.colorbar(scatter, ax=axes[1, 0], label='Rate Adjustment (%)')

# 4. Rate Impact Analysis
rate_impact_by_category = activity_data.groupby('utilization_category')['rate_adjustment'].mean()
rate_impact_by_category = rate_impact_by_category.reindex(existing_categories)

bar_colors = ['darkgreen' if x < 0 else 'darkred' if x > 0 else 'gray' for x in rate_impact_by_category]
bars = axes[1, 1].bar(range(len(rate_impact_by_category)), rate_impact_by_category, 
                      color=bar_colors, alpha=0.7)
axes[1, 1].set_title('Average Rate Adjustment by Category')
axes[1, 1].set_ylabel('Rate Adjustment (%)')
axes[1, 1].set_xticks(range(len(rate_impact_by_category)))
axes[1, 1].set_xticklabels(existing_categories, rotation=45)
axes[1, 1].axhline(y=0, color='black', linestyle='-', alpha=0.5)

# Add value labels on bars
for bar, value in zip(bars, rate_impact_by_category):
    height = bar.get_height()
    axes[1, 1].text(bar.get_x() + bar.get_width()/2., height + (0.01 if height >= 0 else -0.02),
                     f'{value:.3f}%', ha='center', va='bottom' if height >= 0 else 'top')

plt.tight_layout()
plt.show()

## Real-time Activity Monitoring

In [ ]:
# Simulate real-time monitoring
async def simulate_real_time_monitoring(duration_minutes=5):
    """Simulate real-time network monitoring"""
    import random
    
    print(f"🔄 Starting {duration_minutes}-minute real-time monitoring simulation...")
    
    real_time_data = []
    
    for minute in range(duration_minutes):
        # Simulate getting real-time activity score
        current_tps = random.uniform(80, 400)
        mempool_util = random.uniform(0.1, 0.6)
        
        # Mock the real-time activity score calculation
        activity_score = (current_tps / 500) * 0.6 + (1 - mempool_util) * 0.4
        activity_score = min(1.0, max(0.0, activity_score))
        
        # Determine rate adjustment
        if current_tps >= 400:
            rate_adj = -0.05
            category = "high"
        elif current_tps >= 100:
            rate_adj = 0.0
            category = "moderate"
        elif current_tps >= 50:
            rate_adj = 0.05
            category = "low"
        else:
            rate_adj = 0.1
            category = "very_low"
        
        timestamp = datetime.utcnow() + timedelta(minutes=minute)
        
        data_point = {
            'timestamp': timestamp,
            'tps': current_tps,
            'mempool_utilization': mempool_util,
            'activity_score': activity_score,
            'rate_adjustment': rate_adj,
            'category': category
        }
        
        real_time_data.append(data_point)
        
        print(f"⏱️  Minute {minute + 1}: TPS={current_tps:.1f}, Score={activity_score:.3f}, "
              f"Category={category.upper()}, Rate Adj={rate_adj:+.2f}%")
        
        # Simulate 1-minute delay
        await asyncio.sleep(0.1)  # Shortened for demo
    
    return pd.DataFrame(real_time_data)

# Run real-time simulation
real_time_df = await simulate_real_time_monitoring(duration_minutes=5)
print(f"\n✅ Real-time monitoring completed. Collected {len(real_time_df)} data points.")

In [ ]:
# Visualize real-time monitoring results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Real-time Network Monitoring Results', fontsize=16, fontweight='bold')

# 1. TPS over time
axes[0, 0].plot(real_time_df.index, real_time_df['tps'], marker='o', linewidth=2, markersize=6)
axes[0, 0].set_title('Real-time TPS Monitoring')
axes[0, 0].set_ylabel('TPS')
axes[0, 0].set_xlabel('Time (minutes)')
axes[0, 0].grid(True, alpha=0.3)

# Add threshold lines
axes[0, 0].axhline(y=400, color='green', linestyle='--', alpha=0.7, label='High Threshold')
axes[0, 0].axhline(y=100, color='orange', linestyle='--', alpha=0.7, label='Moderate Threshold')
axes[0, 0].axhline(y=50, color='red', linestyle='--', alpha=0.7, label='Low Threshold')
axes[0, 0].legend()

# 2. Activity Score
axes[0, 1].plot(real_time_df.index, real_time_df['activity_score'], 
                marker='s', linewidth=2, markersize=6, color='purple')
axes[0, 1].set_title('Activity Score Tracking')
axes[0, 1].set_ylabel('Activity Score')
axes[0, 1].set_xlabel('Time (minutes)')
axes[0, 1].set_ylim(0, 1)
axes[0, 1].grid(True, alpha=0.3)

# 3. Rate Adjustments
colors = ['green' if x < 0 else 'red' if x > 0 else 'gray' for x in real_time_df['rate_adjustment']]
bars = axes[1, 0].bar(real_time_df.index, real_time_df['rate_adjustment'], color=colors, alpha=0.7)
axes[1, 0].set_title('Rate Adjustment Recommendations')
axes[1, 0].set_ylabel('Rate Adjustment (%)')
axes[1, 0].set_xlabel('Time (minutes)')
axes[1, 0].axhline(y=0, color='black', linestyle='-', alpha=0.5)

# Add value labels
for bar, value in zip(bars, real_time_df['rate_adjustment']):
    height = bar.get_height()
    axes[1, 0].text(bar.get_x() + bar.get_width()/2., height + (0.005 if height >= 0 else -0.01),
                     f'{value:+.2f}%', ha='center', va='bottom' if height >= 0 else 'top', fontsize=9)

# 4. Network Utilization Categories
category_colors = {'very_high': 'darkgreen', 'high': 'green', 'moderate': 'yellow', 
                   'low': 'orange', 'very_low': 'red'}
colors = [category_colors.get(cat, 'gray') for cat in real_time_df['category']]

axes[1, 1].scatter(real_time_df.index, real_time_df['tps'], c=colors, s=100, alpha=0.8)
axes[1, 1].set_title('Utilization Categories')
axes[1, 1].set_ylabel('TPS')
axes[1, 1].set_xlabel('Time (minutes)')

# Create legend for categories
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color, label=cat.title()) 
                   for cat, color in category_colors.items() 
                   if cat in real_time_df['category'].values]
axes[1, 1].legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.show()

## Interest Rate Impact Analysis

In [ ]:
# Analyze the impact of network activity on interest rates
base_rate = 5.0  # 5% base interest rate

# Calculate adjusted rates for historical data
activity_data['base_rate'] = base_rate
activity_data['adjusted_rate'] = activity_data['base_rate'] + activity_data['rate_adjustment']

# Rate statistics
rate_stats = {
    'base_rate': base_rate,
    'avg_adjustment': activity_data['rate_adjustment'].mean(),
    'avg_adjusted_rate': activity_data['adjusted_rate'].mean(),
    'min_rate': activity_data['adjusted_rate'].min(),
    'max_rate': activity_data['adjusted_rate'].max(),
    'rate_volatility': activity_data['adjusted_rate'].std(),
    'favorable_periods': len(activity_data[activity_data['rate_adjustment'] < 0]),
    'unfavorable_periods': len(activity_data[activity_data['rate_adjustment'] > 0]),
    'neutral_periods': len(activity_data[activity_data['rate_adjustment'] == 0])
}

total_periods = len(activity_data)

print("💰 Interest Rate Impact Analysis")
print("=" * 50)
print(f"Base Interest Rate: {rate_stats['base_rate']:.2f}%")
print(f"Average Rate Adjustment: {rate_stats['avg_adjustment']:+.3f}%")
print(f"Average Adjusted Rate: {rate_stats['avg_adjusted_rate']:.3f}%")
print(f"Rate Range: {rate_stats['min_rate']:.3f}% - {rate_stats['max_rate']:.3f}%")
print(f"Rate Volatility (Std Dev): {rate_stats['rate_volatility']:.3f}%")
print("\n📊 Rate Adjustment Distribution:")
print(f"Favorable Periods (rate reduction): {rate_stats['favorable_periods']} ({rate_stats['favorable_periods']/total_periods*100:.1f}%)")
print(f"Neutral Periods (no adjustment): {rate_stats['neutral_periods']} ({rate_stats['neutral_periods']/total_periods*100:.1f}%)")
print(f"Unfavorable Periods (rate increase): {rate_stats['unfavorable_periods']} ({rate_stats['unfavorable_periods']/total_periods*100:.1f}%)")

# Economic impact analysis
if rate_stats['avg_adjustment'] < 0:
    print(f"\n🟢 NET POSITIVE: Network activity supports {abs(rate_stats['avg_adjustment']):.2f}% average rate reduction")
elif rate_stats['avg_adjustment'] > 0:
    print(f"\n🔴 NET NEGATIVE: Low network activity causes {rate_stats['avg_adjustment']:.2f}% average rate increase")
else:
    print(f"\n🟡 NEUTRAL: Network activity has no net impact on rates")

In [ ]:
# Create interest rate impact visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Interest Rate Impact Analysis', fontsize=16, fontweight='bold')

# 1. Rate evolution over time
axes[0, 0].plot(activity_data['timestamp'], activity_data['base_rate'], 
                linewidth=2, color='gray', alpha=0.7, label='Base Rate')
axes[0, 0].plot(activity_data['timestamp'], activity_data['adjusted_rate'], 
                linewidth=2, color='blue', label='Adjusted Rate')
axes[0, 0].fill_between(activity_data['timestamp'], 
                        activity_data['base_rate'], activity_data['adjusted_rate'],
                        alpha=0.3, color='blue')
axes[0, 0].set_title('Interest Rate Evolution')
axes[0, 0].set_ylabel('Interest Rate (%)')
axes[0, 0].legend()
axes[0, 0].tick_params(axis='x', rotation=45)

# 2. Rate adjustment histogram
axes[0, 1].hist(activity_data['rate_adjustment'], bins=15, alpha=0.7, color='skyblue', edgecolor='black')
axes[0, 1].axvline(activity_data['rate_adjustment'].mean(), color='red', linestyle='--', 
                   linewidth=2, label=f'Mean: {activity_data["rate_adjustment"].mean():.3f}%')
axes[0, 1].axvline(0, color='black', linestyle='-', alpha=0.5, label='No Adjustment')
axes[0, 1].set_title('Rate Adjustment Distribution')
axes[0, 1].set_xlabel('Rate Adjustment (%)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].legend()

# 3. Activity score vs rate adjustment correlation
scatter = axes[1, 0].scatter(activity_data['overall_activity_score'], activity_data['rate_adjustment'],
                           c=activity_data['tps'], cmap='viridis', alpha=0.7, s=50)
axes[1, 0].set_xlabel('Overall Activity Score')
axes[1, 0].set_ylabel('Rate Adjustment (%)')
axes[1, 0].set_title('Activity Score vs Rate Adjustment')
axes[1, 0].axhline(y=0, color='black', linestyle='-', alpha=0.5)
plt.colorbar(scatter, ax=axes[1, 0], label='TPS')

# Add trend line
z = np.polyfit(activity_data['overall_activity_score'], activity_data['rate_adjustment'], 1)
p = np.poly1d(z)
axes[1, 0].plot(activity_data['overall_activity_score'], p(activity_data['overall_activity_score']),
                "r--", alpha=0.8, linewidth=2, label=f'Trend: {z[0]:.3f}x + {z[1]:.3f}')
axes[1, 0].legend()

# 4. Cumulative rate impact
activity_data['cumulative_adjustment'] = activity_data['rate_adjustment'].cumsum()
axes[1, 1].plot(activity_data['timestamp'], activity_data['cumulative_adjustment'],
                linewidth=2, color='purple')
axes[1, 1].axhline(y=0, color='black', linestyle='-', alpha=0.5)
axes[1, 1].set_title('Cumulative Rate Impact')
axes[1, 1].set_ylabel('Cumulative Adjustment (%)')
axes[1, 1].tick_params(axis='x', rotation=45)

# Fill areas
axes[1, 1].fill_between(activity_data['timestamp'], 0, activity_data['cumulative_adjustment'],
                        where=(activity_data['cumulative_adjustment'] >= 0), 
                        color='red', alpha=0.3, label='Net Rate Increase')
axes[1, 1].fill_between(activity_data['timestamp'], 0, activity_data['cumulative_adjustment'],
                        where=(activity_data['cumulative_adjustment'] < 0), 
                        color='green', alpha=0.3, label='Net Rate Reduction')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## Predictive Analysis and Recommendations

In [ ]:
# Generate recommendations based on network activity patterns
def generate_network_recommendations(activity_df, rate_stats):
    recommendations = []
    
    # Analyze current trends
    recent_data = activity_df.tail(12)  # Last 3 hours
    recent_avg_score = recent_data['overall_activity_score'].mean()
    recent_avg_tps = recent_data['tps'].mean()
    recent_avg_adjustment = recent_data['rate_adjustment'].mean()
    
    # Overall network health
    if recent_avg_score > 0.8:
        recommendations.append("🟢 Excellent network activity - maintain favorable lending terms")
    elif recent_avg_score > 0.6:
        recommendations.append("🟡 Good network activity - standard lending terms appropriate")
    elif recent_avg_score > 0.4:
        recommendations.append("🟠 Moderate network activity - monitor for improvements")
    else:
        recommendations.append("🔴 Low network activity - consider risk premiums")
    
    # TPS-based recommendations
    if recent_avg_tps > 500:
        recommendations.append("⚡ High throughput detected - network can support increased lending volume")
    elif recent_avg_tps < 100:
        recommendations.append("🐌 Low throughput - may need to limit lending activity during peak times")
    
    # Rate adjustment patterns
    if rate_stats['favorable_periods'] > rate_stats['unfavorable_periods']:
        recommendations.append("📈 Network activity generally supports lower rates - competitive advantage")
    elif rate_stats['unfavorable_periods'] > rate_stats['favorable_periods']:
        recommendations.append("📉 Network activity often requires rate premiums - factor into pricing")
    
    # Volatility recommendations
    if rate_stats['rate_volatility'] > 0.05:
        recommendations.append("🎢 High rate volatility - implement dynamic rate adjustment mechanisms")
    
    # Time-based patterns
    hourly_activity = activity_df.groupby(activity_df['timestamp'].dt.hour)['overall_activity_score'].mean()
    peak_hour = hourly_activity.idxmax()
    low_hour = hourly_activity.idxmin()
    
    recommendations.append(f"⏰ Peak activity at {peak_hour}:00, lowest at {low_hour}:00 - adjust operational hours")
    
    # Future predictions
    trend_slope = np.polyfit(range(len(recent_data)), recent_data['overall_activity_score'], 1)[0]
    if trend_slope > 0.01:
        recommendations.append("📊 Positive activity trend - expect improving rate conditions")
    elif trend_slope < -0.01:
        recommendations.append("📊 Declining activity trend - prepare for potential rate increases")
    
    return recommendations

# Generate recommendations
recommendations = generate_network_recommendations(activity_data, rate_stats)

print("💡 Network Activity-Based Recommendations")
print("=" * 60)
for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec}")

# Current network status summary
current_status = {
    'current_tps': activity_data['tps'].iloc[-1],
    'current_score': activity_data['overall_activity_score'].iloc[-1],
    'current_category': activity_data['utilization_category'].iloc[-1],
    'current_adjustment': activity_data['rate_adjustment'].iloc[-1],
    'current_rate': activity_data['adjusted_rate'].iloc[-1]
}

print(f"\n📊 Current Network Status")
print("=" * 30)
print(f"🚀 Current TPS: {current_status['current_tps']:.1f}")
print(f"📈 Activity Score: {current_status['current_score']:.3f}")
print(f"🏷️  Category: {current_status['current_category'].upper()}")
print(f"💰 Rate Adjustment: {current_status['current_adjustment']:+.2f}%")
print(f"🎯 Effective Rate: {current_status['current_rate']:.3f}%")

## Performance Dashboard

In [ ]:
# Create a comprehensive performance dashboard
print("📊 ALGORAND NETWORK ACTIVITY DASHBOARD")
print("=" * 80)

# Network Health Section
print("\n🌐 NETWORK HEALTH METRICS")
print("-" * 40)
avg_tps = activity_data['tps'].mean()
avg_participation = activity_data['participation_rate'].mean()
avg_block_time = activity_data['block_time'].mean()
avg_mempool = activity_data['mempool_utilization'].mean()

print(f"Average TPS: {avg_tps:.1f} {'🟢' if avg_tps > 200 else '🟡' if avg_tps > 100 else '🔴'}")
print(f"Consensus Participation: {avg_participation:.1%} {'🟢' if avg_participation > 0.9 else '🟡' if avg_participation > 0.85 else '🔴'}")
print(f"Average Block Time: {avg_block_time:.1f}s {'🟢' if abs(avg_block_time - 4.5) < 0.5 else '🟡' if abs(avg_block_time - 4.5) < 1.0 else '🔴'}")
print(f"Mempool Utilization: {avg_mempool:.1%} {'🟢' if avg_mempool < 0.3 else '🟡' if avg_mempool < 0.6 else '🔴'}")

# DApp Ecosystem Section
print("\n🏗️ DAPP ECOSYSTEM ACTIVITY")
print("-" * 40)
avg_defi_volume = activity_data['defi_volume_24h'].mean()
avg_dapp_calls = activity_data['dapp_calls'].mean()
avg_nft_sales = activity_data['nft_sales'].mean()
avg_governance = activity_data['governance_votes'].mean()

print(f"Average DeFi Volume: ${avg_defi_volume:,.0f} {'🟢' if avg_defi_volume > 2000000 else '🟡' if avg_defi_volume > 1000000 else '🔴'}")
print(f"Daily DApp Calls: {avg_dapp_calls:,.0f} {'🟢' if avg_dapp_calls > 20000 else '🟡' if avg_dapp_calls > 10000 else '🔴'}")
print(f"NFT Sales: {avg_nft_sales:.0f}/day {'🟢' if avg_nft_sales > 100 else '🟡' if avg_nft_sales > 50 else '🔴'}")
print(f"Governance Votes: {avg_governance:.0f} {'🟢' if avg_governance > 500 else '🟡' if avg_governance > 200 else '🔴'}")

# Rate Impact Section
print("\n💰 INTEREST RATE IMPACT")
print("-" * 40)
print(f"Base Rate: {base_rate:.2f}%")
print(f"Average Adjustment: {rate_stats['avg_adjustment']:+.3f}%")
print(f"Current Effective Rate: {current_status['current_rate']:.3f}%")
print(f"Rate Range: {rate_stats['min_rate']:.2f}% - {rate_stats['max_rate']:.2f}%")

# Activity Score Summary
print("\n📈 ACTIVITY SCORE BREAKDOWN")
print("-" * 40)
avg_network_score = activity_data['network_score'].mean()
avg_dapp_score = activity_data['dapp_score'].mean()
avg_overall_score = activity_data['overall_activity_score'].mean()

print(f"Network Score: {avg_network_score:.3f} {'🟢' if avg_network_score > 0.7 else '🟡' if avg_network_score > 0.5 else '🔴'}")
print(f"DApp Score: {avg_dapp_score:.3f} {'🟢' if avg_dapp_score > 0.7 else '🟡' if avg_dapp_score > 0.5 else '🔴'}")
print(f"Overall Score: {avg_overall_score:.3f} {'🟢' if avg_overall_score > 0.7 else '🟡' if avg_overall_score > 0.5 else '🔴'}")

# Utilization Summary
print("\n⚡ NETWORK UTILIZATION SUMMARY")
print("-" * 40)
for category in ['very_high', 'high', 'moderate', 'low', 'very_low']:
    count = len(activity_data[activity_data['utilization_category'] == category])
    if count > 0:
        percentage = (count / len(activity_data)) * 100
        rate_impact = config['utilization_categories'][category]['rate_impact']
        emoji = {'very_high': '🚀', 'high': '⚡', 'moderate': '🔄', 'low': '🐌', 'very_low': '😴'}[category]
        print(f"{emoji} {category.upper():<10}: {percentage:>5.1f}% ({count:>2} periods) Rate: {rate_impact:+.2f}%")

print(f"\n{'='*80}")
print(f"📊 DASHBOARD GENERATED: {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC")
print(f"🔄 DATA POINTS ANALYZED: {len(activity_data)}")
print(f"⏱️  TIME PERIOD: 24 hours")
print(f"📈 TRENDING: {'UP' if trend_slope > 0 else 'DOWN' if trend_slope < 0 else 'STABLE'}")

## Conclusion

This demonstration showcases the comprehensive Algorand Network Activity Monitor for dynamic interest rate determination:

### Key Capabilities:
1. **Real-time Network Monitoring** - TPS, block times, consensus participation, mempool status
2. **DApp Ecosystem Analysis** - DeFi volume, smart contract activity, NFT marketplace engagement
3. **Activity-Based Rate Adjustments** - Dynamic interest rate modifications based on network utilization
4. **Predictive Analytics** - Trend analysis and forecasting for proactive rate management
5. **Comprehensive Dashboard** - Real-time monitoring and historical analysis

### Benefits for Lending Protocols:
- **Dynamic Pricing** - Rates adjust automatically based on network health and activity
- **Market Responsiveness** - Quick adaptation to changing network conditions
- **Risk Management** - Higher rates during network stress, lower rates during optimal performance
- **Competitive Advantage** - Data-driven rate optimization
- **Transparency** - Clear metrics and rationale for rate adjustments

### Integration Architecture:
1. **MCP Services** - Real-time blockchain data streaming
2. **Multi-source Data** - Node metrics, indexer data, DEX APIs
3. **Automated Decision Making** - Rule-based rate adjustments
4. **Monitoring & Alerting** - Threshold-based notifications
5. **Historical Analysis** - Pattern recognition and trend forecasting

### Rate Adjustment Logic:
- **Very High Activity (800+ TPS)**: -10 basis points (reward high usage)
- **High Activity (400+ TPS)**: -5 basis points (encourage usage)
- **Moderate Activity (100+ TPS)**: No adjustment (neutral)
- **Low Activity (50+ TPS)**: +5 basis points (risk premium)
- **Very Low Activity (<50 TPS)**: +10 basis points (higher risk premium)

This system enables lending protocols to dynamically optimize interest rates based on real-time Algorand network conditions, providing both competitive advantages and risk management benefits.